In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error
import time

import warnings
warnings.filterwarnings('ignore')

# импорт tabkanet 
from tabkanet.models.tabkanet import TabKANet
from tabkanet.tools_regression import get_dataset, get_data_loader, seed_everything, train


In [54]:
import math
import time
import gc
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

import shap

# Проверка GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [14]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
import xgboost as xgb
import catboost as cb

In [19]:
HP = {
    'embedding_dim': 64,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'attn_dropout_rate': 0.1,
    'mlp_hidden_dims': [32],
    'activation': 'relu',
    'ffn_dropout_rate': 0.1,
    'batch_size': 128,
    'epochs': 50,
    'seed': 42,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}
seed_everything(HP['seed'])
print(f"Using device: {HP['device']}")

Using device: cuda


In [ ]:
# model = TabKANet(
#             output_dim=dataset_config['output_dim'],
#             vocabulary=vocab,
#             num_continuous_features=len(dataset_config['continuous']),
#             embedding_dim=hp_local['embedding_dim'],
#             nhead=hp_local['nhead'],
#             num_layers=hp_local['num_layers'],
#             dim_feedforward=hp_local['dim_feedforward'],
#             attn_dropout_rate=hp_local['attn_dropout_rate'],
#             mlp_hidden_dims=hp_local['mlp_hidden_dims'],
#             activation=hp_local['activation'],
#             ffn_dropout_rate=hp_local['ffn_dropout_rate'],
#             transformer_type='standard',
#             kan_hidden_dim=None
#         ).to(hp_local['device'])
# params = sum(p.numel() for p in model.parameters())
# print(params)
# model = TabKANet(
#             output_dim=dataset_config['output_dim'],
#             vocabulary=vocab,
#             num_continuous_features=len(dataset_config['continuous']),
#             embedding_dim=hp_local['embedding_dim'],
#             nhead=hp_local['nhead'],
#             num_layers=hp_local['num_layers'],
#             dim_feedforward=hp_local['dim_feedforward'],
#             attn_dropout_rate=hp_local['attn_dropout_rate'],
#             mlp_hidden_dims=hp_local['mlp_hidden_dims'],
#             activation=hp_local['activation'],
#             ffn_dropout_rate=hp_local['ffn_dropout_rate'],
#             transformer_type='kan',
#             kan_hidden_dim=None
#         ).to(hp_local['device'])
# params = sum(p.numel() for p in model.parameters())
# print(params)

# model = TabKANet(
#             output_dim=dataset_config['output_dim'],
#             vocabulary=vocab,
#             num_continuous_features=len(dataset_config['continuous']),
#             embedding_dim=hp_local['embedding_dim'],
#             nhead=hp_local['nhead'],
#             num_layers=hp_local['num_layers'],
#             dim_feedforward=hp_local['dim_feedforward'],
#             attn_dropout_rate=hp_local['attn_dropout_rate'],
#             mlp_hidden_dims=hp_local['mlp_hidden_dims'],
#             activation=hp_local['activation'],
#             ffn_dropout_rate=hp_local['ffn_dropout_rate'],
#             transformer_type='kan',
#             kan_hidden_dim=16
#         ).to(hp_local['device'])
# params = sum(p.numel() for p in model.parameters())
# print(params)

388538
830202
400122


In [35]:
import torch
import torch.nn as nn
import numpy as np
from copy import deepcopy

def prune_kan_basis(model, pruning_ratio=0.2):
    """
    Обнуляет наименее важные B-сплайновые базисные функции в KAN-слоях.
    Важность = L2-норма по всем входам и выходам.
    """
    pruned = 0
    total = 0
    for name, module in model.named_modules():
        if hasattr(module, 'spline_weight'):  # KANLinear
            weight = module.spline_weight  # shape: (out, in, coeff)
            # Вычисляем важность для каждой базисной функции (по оси coeff)
            importance = torch.norm(weight, p=2, dim=(0, 1))  # (coeff,)
            total += importance.numel()
            threshold = torch.quantile(importance, pruning_ratio)
            mask = (importance > threshold).float()  # 1 – оставить, 0 – обнулить
            # Применяем маску к spline_weight и base_weight
            module.spline_weight.data *= mask[None, None, :]  # расширяем размерности
            # base_weight не трогаем, т.к. он не связан с базисными функциями
            pruned += (mask == 0).sum().item()
    print(f"Pruned {pruned} basis functions out of {total} ({(pruned/total)*100:.2f}%)")
    return model

def prune_kan_neurons(model, pruning_ratio=0.2):
    """
    Обнуляет целые нейроны (выходные размерности) в KAN-слоях.
    Важность нейрона = L2-норма всех его сплайнов.
    """
    for name, module in model.named_modules():
        if hasattr(module, 'spline_weight'):  # KANLinear
            weight = module.spline_weight  # (out, in, coeff)
            # Важность для каждого выходного нейрона: норма по in и coeff
            importance = torch.norm(weight, p=2, dim=(1, 2))  # (out,)
            threshold = torch.quantile(importance, pruning_ratio)
            mask = (importance > threshold).float()  # (out,)
            # Обнуляем строки spline_weight и base_weight
            module.spline_weight.data *= mask[:, None, None]
            module.base_weight.data *= mask[:, None]
            # Если есть spline_scaler, тоже обнуляем
            if hasattr(module, 'spline_scaler'):
                module.spline_scaler.data *= mask[:, None]
    return model

def prune_mlp_structured(model, pruning_ratio=0.2):
    """
    Структурный прунинг для MLP-слоёв: удаляет нейроны с наименьшей L1-нормой.
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.out_features > 1:
            # Важность нейронов = L1-норма весов, входящих в нейрон
            importance = torch.norm(module.weight, p=1, dim=1)  # (out,)
            threshold = torch.quantile(importance, pruning_ratio)
            mask = (importance > threshold).float()  # (out,)
            # Обнуляем веса и bias для этих нейронов
            module.weight.data *= mask[:, None]
            if module.bias is not None:
                module.bias.data *= mask
    return model

def prune_mlp_unstructured(model, pruning_ratio=0.2):
    """
    Неструктурный прунинг: обнуляет отдельные веса с наименьшей абсолютной величиной.
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            weight = module.weight.data
            importance = torch.abs(weight).flatten()
            threshold = torch.quantile(importance, pruning_ratio)
            mask = (torch.abs(weight) > threshold).float()
            module.weight.data *= mask
    return model

def apply_pruning(model, pruning_type='kan_basis', pruning_ratio=0.2):
    """
    Универсальная функция применения прунинга.
    Поддерживаемые типы: 'kan_basis', 'kan_neuron', 'mlp_structured', 'mlp_unstructured'.
    """
    model_copy = deepcopy(model)  # чтобы не портить оригинал
    if pruning_type == 'kan_basis':
        prune_kan_basis(model_copy, pruning_ratio)
    elif pruning_type == 'kan_neuron':
        prune_kan_neurons(model_copy, pruning_ratio)
    elif pruning_type == 'mlp_structured':
        prune_mlp_structured(model_copy, pruning_ratio)
    elif pruning_type == 'mlp_unstructured':
        prune_mlp_unstructured(model_copy, pruning_ratio)
    else:
        raise ValueError(f"Unsupported pruning_type: {pruning_type}")
    return model_copy

def fine_tune_model(model, train_loader, val_loader, test_loader, task, output_dim, epochs=5, lr=1e-4, device='cuda'):
    """
    Дообучает модель после прунинга с низкой скоростью обучения.
    """
    if task == 'regression':
        criterion = nn.MSELoss()
        train_task = 'regression'
    else:
        criterion = nn.CrossEntropyLoss()
        train_task = 'classification'
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    if task == 'multiclass':
        custom_metric = lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro')
    else:
        custom_metric = None   # для бинарной классификации можно передать функцию для AUC
        
    train_loss_hist, val_loss_hist, _ = train(
        model, epochs=epochs, task=train_task,
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        optimizer=optimizer, criterion=criterion,
        scheduler=scheduler, custom_metric=custom_metric,
        gpu_num=0 if torch.cuda.is_available() else -1,
        save_model_path=None
    )
    return model

def evaluate_pruned_model(model, test_loader, task, device):
    """
    Оценка модели на тесте.
    """
    return evaluate_model(model, test_loader, task, device)  # используем уже существующую функцию

## Final pruning experiment


In [56]:
def get_dataset_config(dataset_name):
    """Возвращает конфигурацию для заданного датасета (аналог DATASET_CONFIG)."""
    # Предполагаем, что у вас уже определён словарь DATASET_CONFIG.
    # Если нет – задайте его вручную.
    # Здесь для примера возвращаем конфигурацию для multi_seg.
    configs = {
         'bankmarketing': {
        'folder': 'bankmarketing',
        'target': 'Output',
        'continuous': ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous'],
        'categorical': ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day', 'month', 'poutcome'],
        'task': 'binary',
        'output_dim': 2,
        'metric': 'auc'
    },
    'onlineshoper': {
        'folder': 'onlineshoper',
        'target': 'Revenue',
        'continuous': ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues'],
        'categorical': ['Administrative', 'Informational', 'ProductRelated', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend'],
        'task': 'binary',
        'output_dim': 2,
        'metric': 'auc'
    },
    'cahouse': {
        'folder': 'cahouse',
        'target': 'Output',
        'continuous': ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'],
        'categorical': ['ocean_proximity'],
        'task': 'regression',
        'output_dim': 1,
        'metric': 'rmse'
    },
    'multi_forest': {
        'folder': 'multi_forest',
        'target': 'class',
        'continuous': ['elevation', 'aspect', 'slope', 'horizontal_distance_to_hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points'],
        'categorical': ['wilderness_area1', 'wilderness_area2', 'wilderness_area3', 'wilderness_area4', 'soil_type_1', 'soil_type_2', 'soil_type_3', 'soil_type_4', 'soil_type_5', 'soil_type_6', 'soil_type_7', 'soil_type_8', 'soil_type_9', 'soil_type_10', 'soil_type_11', 'soil_type_12', 'soil_type_13', 'soil_type_14', 'soil_type_15', 'soil_type_16', 'soil_type_17', 'soil_type_18', 'soil_type_19', 'soil_type_20', 'soil_type_21', 'soil_type_22', 'soil_type_23', 'soil_type_24', 'soil_type_25', 'soil_type_26', 'soil_type_27', 'soil_type_28', 'soil_type_29', 'soil_type_30', 'soil_type_31', 'soil_type_32', 'soil_type_33', 'soil_type_34', 'soil_type_35', 'soil_type_36', 'soil_type_37', 'soil_type_38', 'soil_type_39', 'soil_type_40'],
        'task': 'multiclass',
        'output_dim': 7,
        'metric': 'f1'
    },
    'multi_seg': {
        'folder': 'multi_seg',
        'target': 'class',
        'continuous': ['region.centroid.col', 'region.centroid.row', 'region.pixel.count', 'short.line.density.5', 'short.line.density.2', 'vedge.mean', 'vegde.sd', 'hedge.mean', 'hedge.sd', 'intensity.mean', 'rawred.mean', 'rawblue.mean', 'rawgreen.mean', 'exred.mean', 'exblue.mean', 'exgreen.mean', 'value.mean', 'saturation.mean', 'hue.mean'],
        'categorical': [],
        'task': 'multiclass',
        'output_dim': 7,
        'metric': 'f1'
    },
    'cpu_small': {
        'folder': 'cpu_small',
        'target': 'usr',
        'continuous': ['lread', 'lwrite', 'scall', 'sread', 'swrite', 'fork', 'exec', 'rchar', 'wchar', 'runqsz', 'freemem', 'freeswap'],
        'categorical': [],
        'task': 'regression',
        'output_dim': 1,
        'metric': 'rmse'
    },
    'sarcos': {
        'folder': 'sarcos',
        'target': 'V22',
        'continuous': ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28'],
        'categorical': [],
        'task': 'regression',
        'output_dim': 1,
        'metric': 'rmse'
    }
    }
    return configs[dataset_name]

In [57]:
def load_fold_data(dataset_name, fold, base_path="templates"):
    config = get_dataset_config(dataset_name)
    folder = config["folder"]
    target = config["target"]

    train_df = pd.read_csv(Path(base_path) / folder / f"Fold{fold}" / "train.csv").fillna("EMPTY")
    val_df   = pd.read_csv(Path(base_path) / folder / f"Fold{fold}" / "val.csv").fillna("EMPTY")
    test_df  = pd.read_csv(Path(base_path) / folder / f"Fold{fold}" / "test.csv").fillna("EMPTY")

    # Кодируем target
    if train_df[target].dtype == object:
        le = LabelEncoder()
        all_targets = pd.concat([train_df[target], val_df[target], test_df[target]])
        le.fit(all_targets)
        train_df[target] = le.transform(train_df[target])
        val_df[target]   = le.transform(val_df[target])
        test_df[target]  = le.transform(test_df[target])

    cat_cols = config["categorical"]
    cont_cols = config["continuous"]

    # Строим vocabulary на основе всех данных (train+val+test)
    vocabulary = {}
    if cat_cols:
        for col in cat_cols:
            all_vals = pd.concat([train_df[col], val_df[col], test_df[col]]).unique()
            all_vals = np.sort(all_vals.astype(str))  # приводим к строке для единообразия
            vocabulary[col] = {str(v): i for i, v in enumerate(all_vals)}

    # Кодируем категориальные с помощью маппинга vocabulary
    def encode_cat(df, vocab):
        encoded = np.zeros((len(df), len(vocab)), dtype=np.int64)
        for i, col in enumerate(vocab.keys()):
            # Используем .map с заменой неизвестных на 0 (можно использовать -1, но тогда нужно избегать отрицательных индексов)
            encoded[:, i] = df[col].astype(str).map(vocab[col]).fillna(0).values
        return encoded

    if cat_cols:
        X_train_cat = encode_cat(train_df, vocabulary)
        X_val_cat   = encode_cat(val_df, vocabulary)
        X_test_cat  = encode_cat(test_df, vocabulary)
    else:
        X_train_cat = np.empty((len(train_df), 0), dtype=np.int64)
        X_val_cat   = np.empty((len(val_df), 0), dtype=np.int64)
        X_test_cat  = np.empty((len(test_df), 0), dtype=np.int64)

    X_train_cont = train_df[cont_cols].values.astype(np.float32)
    X_val_cont   = val_df[cont_cols].values.astype(np.float32)
    X_test_cont  = test_df[cont_cols].values.astype(np.float32)

    return {
        "X_train_cat": X_train_cat,
        "X_train_cont": X_train_cont,
        "y_train": train_df[target].values,
        "X_val_cat": X_val_cat,
        "X_val_cont": X_val_cont,
        "y_val": val_df[target].values,
        "X_test_cat": X_test_cat,
        "X_test_cont": X_test_cont,
        "y_test": test_df[target].values,
        "vocabulary": vocabulary,
        "num_continuous": len(cont_cols),
        "output_dim": config["output_dim"],
        "task": config["task"]
    }

In [58]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        cat, cont, y = batch
        cat = cat.to(device)
        cont = cont.to(device)
        y = y.to(device).float() if model.output_dim == 1 else y.to(device).long()
        optimizer.zero_grad()
        pred = model(cat, cont)
        if model.output_dim == 1:
            loss = criterion(pred.squeeze(), y)
        else:
            loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_model(model, loader, device, task="multiclass"):
    model.eval()
    preds_list = []
    targets_list = []
    with torch.no_grad():
        for batch in loader:
            cat, cont, y = batch
            cat = cat.to(device)
            cont = cont.to(device)
            pred = model(cat, cont)
            if task == "binary":
                if pred.size(1) == 1:
                    pred = pred.squeeze()
                else:
                    pred = pred[:, 1]  # вероятности для положительного класса
                preds_list.append(pred.cpu().numpy())
            elif task == "multiclass":
                preds_list.append(torch.argmax(pred, dim=1).cpu().numpy())
            else:  # regression
                preds_list.append(pred.squeeze().cpu().numpy())
            targets_list.append(y.cpu().numpy())
    all_preds = np.concatenate(preds_list)
    all_targets = np.concatenate(targets_list)

    if task == "binary":
        # используем ROC-AUC
        if len(np.unique(all_targets)) == 2:
            return roc_auc_score(all_targets, all_preds)
        else:
            return accuracy_score(all_targets, (all_preds > 0.5).astype(int))
    elif task == "multiclass":
        return f1_score(all_targets, all_preds, average="macro")
    else:
        return np.sqrt(mean_squared_error(all_targets, all_preds))

In [59]:
from tqdm import trange
import torch.optim.lr_scheduler as lr_scheduler

def train_and_evaluate_model(
    model,
    train_data,
    val_data,
    test_data,
    batch_size=256,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    early_stopping_patience=50,
    device="cpu",
    metric_mode='max'          # 'max' или 'min'
):
    # Создаём DataLoader'ы
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if model.output_dim == 1:
        criterion = nn.MSELoss() if model.task == "regression" else nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    # Добавляем планировщик
    scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer, mode=metric_mode, factor=0.5, patience=3
    )

    best_val_metric = -np.inf if metric_mode == 'max' else np.inf
    patience_counter = 0
    best_model_state = None

    for epoch in trange(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_metric = evaluate_model(model, val_loader, device, task=model.task)

        # Обновляем scheduler на основе валидационной метрики
        scheduler.step(val_metric)

        # Сохраняем лучшую модель
        is_better = (metric_mode == 'max' and val_metric > best_val_metric) or \
                    (metric_mode == 'min' and val_metric < best_val_metric)
        if is_better:
            best_val_metric = val_metric
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Восстанавливаем лучшую модель
    model.load_state_dict(best_model_state)
    model.to(device)

    # Оценка на тесте
    test_metric = evaluate_model(model, test_loader, device, task=model.task)
    return test_metric, best_val_metric

In [62]:
def run_full_experiment(dataset_name, lr=5e-2):
    """
    Полный эксперимент для одного датасета:
    - для каждого фолда обучается 2 модели, сразу после обучения применяются все типы прунинга с дообучением
    - результаты собираются по каждому фолду и агрегируются в конце
    """
    dataset_config = get_dataset_config(dataset_name)
    print(f"\n{'='*70}")
    print(f"FULL EXPERIMENT FOR: {dataset_name}")
    print(f"Task: {dataset_config['task']}, Output dim: {dataset_config['output_dim']}")
    print(f"{'='*70}")

    # Создаём папку для сохранения результатов
    save_root = Path("saved_models") / dataset_name
    save_root.mkdir(parents=True, exist_ok=True)

    models_to_test = [
        ('standard', None),
        ('kan', 16),
    ]

    pruning_types = [
        'kan_basis',
        'kan_neuron',
        'mlp_structured',
        'mlp_unstructured',
    ]
    PRUNING_RATIO = 0.2
    FT_EPOCHS = 10
    FT_LR = lr/10
    BATCH_SIZE = 256
    WEIGHT_DECAY = 1e-5
    EARLY_STOPPING = 50

    all_results = []

    for model_type, kan_hidden_dim in models_to_test:
        model_name = f"{model_type}_{kan_hidden_dim}" if kan_hidden_dim else model_type
        print(f"\n--- PROCESSING {model_name} ---")

        for fold in range(1, 6):
            print(f"  Fold {fold}...")

            # 1. Загрузка данных для фолда
            data = load_fold_data(dataset_name, fold, base_path="templates")
            vocabulary = data['vocabulary']
            num_cont = data['num_continuous']

            # Подготовка тензоров
            X_train_cat = torch.tensor(data['X_train_cat'], dtype=torch.long)
            X_train_cont = torch.tensor(data['X_train_cont'], dtype=torch.float32)
            y_train = torch.tensor(data['y_train'], dtype=torch.long if data['task'] == 'multiclass' else torch.float32)
            X_val_cat = torch.tensor(data['X_val_cat'], dtype=torch.long)
            X_val_cont = torch.tensor(data['X_val_cont'], dtype=torch.float32)
            y_val = torch.tensor(data['y_val'], dtype=torch.long if data['task'] == 'multiclass' else torch.float32)
            X_test_cat = torch.tensor(data['X_test_cat'], dtype=torch.long)
            X_test_cont = torch.tensor(data['X_test_cont'], dtype=torch.float32)
            y_test = torch.tensor(data['y_test'], dtype=torch.long if data['task'] == 'multiclass' else torch.float32)

            train_ds = TensorDataset(X_train_cat, X_train_cont, y_train)
            val_ds   = TensorDataset(X_val_cat, X_val_cont, y_val)
            test_ds  = TensorDataset(X_test_cat, X_test_cont, y_test)

            # 2. Создаём модель
            device = HP['device']
            model = TabKANet(
                output_dim=dataset_config['output_dim'],
                vocabulary=vocabulary,
                num_continuous_features=num_cont,
                embedding_dim=HP['embedding_dim'],
                nhead=HP['nhead'],
                num_layers=HP['num_layers'],
                dim_feedforward=HP['dim_feedforward'],
                attn_dropout_rate=HP['attn_dropout_rate'],
                mlp_hidden_dims=HP['mlp_hidden_dims'],
                activation=HP['activation'],
                ffn_dropout_rate=HP['ffn_dropout_rate'],
                transformer_type=model_type,
                kan_hidden_dim=kan_hidden_dim
            ).to(device)
            model.task = dataset_config['task']
            model.output_dim = dataset_config['output_dim']

            # 3. Обучение с нуля (получаем baseline метрику и обученную модель)
            print("    Training...")
            metric_mode = 'max' if dataset_config['task'] in ['binary', 'multiclass'] else 'min'
            baseline_metric, best_val_metric = train_and_evaluate_model(
                model,
                train_data=train_ds,
                val_data=val_ds,
                test_data=test_ds,
                batch_size=BATCH_SIZE,
                epochs=100,               # достаточно большое число, будет early stopping
                lr=lr,                    # используем переданный lr
                weight_decay=WEIGHT_DECAY,
                early_stopping_patience=EARLY_STOPPING,
                device=device,
                metric_mode=metric_mode
            )
            print(f"    Baseline test metric: {baseline_metric:.4f}")

            # 4. Применяем прунинг для каждого типа
            for prune_type in pruning_types:
                print(f"    Pruning: {prune_type}...")
                model_copy = deepcopy(model)

                try:
                    pruned_model = apply_pruning(model_copy, pruning_type=prune_type, pruning_ratio=PRUNING_RATIO)
                except Exception as e:
                    print(f"      Error applying {prune_type}: {e}")
                    continue

                # Дообучение после прунинга
                print(f"      Fine-tuning...")
                post_prune_metric, _ = train_and_evaluate_model(
                    pruned_model,
                    train_data=train_ds,
                    val_data=val_ds,
                    test_data=test_ds,
                    batch_size=BATCH_SIZE,
                    epochs=FT_EPOCHS,
                    lr=FT_LR,
                    weight_decay=WEIGHT_DECAY,
                    early_stopping_patience=5,   # небольшое patience для дообучения
                    device=device,
                    metric_mode=metric_mode
                )

                all_results.append({
                    'dataset': dataset_name,
                    'fold': fold,
                    'model': model_name,
                    'pruning_type': prune_type,
                    'pruning_ratio': PRUNING_RATIO,
                    'baseline_metric': baseline_metric,
                    'post_prune_metric': post_prune_metric,
                    'metric_change': post_prune_metric - baseline_metric,
                })

                del pruned_model
                torch.cuda.empty_cache()

            del model
            torch.cuda.empty_cache()
            print(f"    Fold {fold} completed.")

    # ========== АГРЕГАЦИЯ РЕЗУЛЬТАТОВ ==========
    df = pd.DataFrame(all_results)
    if len(df) > 0:
        df.to_csv(f"full_results_{dataset_name}.csv", index=False)
        summary = df.groupby(['model', 'pruning_type']).agg({
            'baseline_metric': ['mean', 'std'],
            'post_prune_metric': ['mean', 'std'],
            'metric_change': ['mean', 'std']
        }).round(4)
        summary.to_csv(f"full_summary_{dataset_name}.csv")
        print(f"\n{'='*70}")
        print(f"SUMMARY FOR {dataset_name}")
        print(summary)
        print(f"\nResults saved to full_results_{dataset_name}.csv and full_summary_{dataset_name}.csv")
    else:
        print(f"No results collected for {dataset_name}")

    return df, summary

In [63]:
binary_datasets = [
    ['bankmarketing', 1e-3],
    ['onlineshoper', 1e-3],
    ['multi_seg', 1e-3],
    ['multi_forest', 1e-3],
    ['cahouse', 5e-2],
    ['cpu_small', 5e-2],
    ['sarcos', 5e-2]
]
for ds, lr in binary_datasets:
     run_full_experiment(ds, lr)

## Загрузка данных для визуализации

In [ ]:
df_pruning = pd.DataFrame(all_pruning_results)
df_pruning.to_csv("pruning_results.csv", index=False)

# Сводная таблица по датасетам и моделям
summary_pruning = df_pruning.groupby(['dataset', 'model', 'pruning_type']).agg({
    'baseline_metric': 'mean',
    'post_prune_metric': 'mean',
    'metric_change': ['mean', 'std']
}).round(4)
summary_pruning.to_csv("pruning_summary.csv")


In [12]:
import torch
import numpy as np
from copy import deepcopy
from pathlib import Path
import pandas as pd

# ============================================================
# Конфигурация
# ============================================================
device = torch.device('cpu')
dataset_name = 'multi_forest'
fold = 1
model_type = 'kan_16'

dataset_config = DATASET_CONFIG[dataset_name]
vocabulary, train_loader, val_loader, test_loader = load_data_for_fold(dataset_config, fold)

if model_type == 'standard':
    trans_type = 'standard'
    kan_hidden = None
elif model_type == 'kan_16':
    trans_type = 'kan'
    kan_hidden = 16
else:
    raise ValueError(f"Unknown model type: {model_type}")

model_path = Path("saved_models") / dataset_name / f"{model_type}_fold{fold}.pth"
if not model_path.exists():
    raise FileNotFoundError(f"Model not found: {model_path}")

# ============================================================
# Загрузка модели на CPU
# ============================================================
model = TabKANet(
    output_dim=dataset_config['output_dim'],
    vocabulary=vocabulary,
    num_continuous_features=len(dataset_config['continuous']),
    embedding_dim=HP['embedding_dim'],
    nhead=HP['nhead'],
    num_layers=HP['num_layers'],
    dim_feedforward=HP['dim_feedforward'],
    attn_dropout_rate=HP['attn_dropout_rate'],
    mlp_hidden_dims=HP['mlp_hidden_dims'],
    activation=HP['activation'],
    ffn_dropout_rate=HP['ffn_dropout_rate'],
    transformer_type=trans_type,
    kan_hidden_dim=kan_hidden
).to(device)
model.load_state_dict(torch.load(model_path, map_location=device), strict=False)
model.eval()
print("✅ Model loaded on CPU")

# ============================================================
# 1. KAN-энкодер непрерывных признаков (первый слой)
# ============================================================
enc_layer = model.encoders['continuous_feature_encoder'].kanclassifier.layers[0]
in_features_enc = enc_layer.in_features
print(f"Encoder KAN layer: in_features={in_features_enc}, out_features={enc_layer.out_features}")

n_points = 200
x_vals = np.linspace(-3, 3, n_points)
x_batch = torch.tensor([[x] * in_features_enc for x in x_vals], dtype=torch.float32, device=device)

with torch.no_grad():
    out_before_enc = enc_layer(x_batch)
y_before_enc = out_before_enc[:, 0].cpu().numpy()

# ============================================================
# 2. Первый KAN-слой в Transformer (FFN), если модель KAN
# ============================================================
if trans_type == 'kan':
    # Берём первый блок трансформера
    first_block = model.transformer_model.blocks[0]
    # Внутри блока есть kan_ffn, внутри которого layers[0] – первый KANLinear
    kan_ffn_layer = first_block.kan_ffn.layers[0]
    in_features_ffn = kan_ffn_layer.in_features
    print(f"Transformer FFN KAN layer: in_features={in_features_ffn}, out_features={kan_ffn_layer.out_features}")

    # Для FFN входная размерность равна embedding_dim (64), но мы не можем подать один скаляр,
    # потому что KAN-слой принимает вектор размерности in_features_ffn.
    # Мы будем подавать вектор, где все элементы равны x (как и для энкодера)
    x_batch_ffn = torch.tensor([[x] * in_features_ffn for x in x_vals], dtype=torch.float32, device=device)

    with torch.no_grad():
        out_before_ffn = kan_ffn_layer(x_batch_ffn)
    y_before_ffn = out_before_ffn[:, 0].cpu().numpy()
else:
    y_before_ffn = None
    print("Standard transformer has no KAN FFN, skipping.")

# ============================================================
# Прунинг (kan_basis) – применяем к полной модели
# ============================================================
pruned_model = apply_pruning(deepcopy(model), pruning_type='kan_basis', pruning_ratio=0.2)
pruned_model.eval()

# ============================================================
# После прунинга – энкодер
# ============================================================
pruned_enc_layer = pruned_model.encoders['continuous_feature_encoder'].kanclassifier.layers[0]
with torch.no_grad():
    out_after_enc = pruned_enc_layer(x_batch)
y_after_enc = out_after_enc[:, 0].cpu().numpy()

# ============================================================
# После прунинга – FFN (если есть)
# ============================================================
if trans_type == 'kan':
    pruned_first_block = pruned_model.transformer_model.blocks[0]
    pruned_kan_ffn_layer = pruned_first_block.kan_ffn.layers[0]
    with torch.no_grad():
        out_after_ffn = pruned_kan_ffn_layer(x_batch_ffn)
    y_after_ffn = out_after_ffn[:, 0].cpu().numpy()
else:
    y_after_ffn = None

# ============================================================
# Сохраняем данные в CSV
# ============================================================
df_dict = {'x': x_vals}
df_dict['encoder_before'] = y_before_enc
df_dict['encoder_after'] = y_after_enc
if y_before_ffn is not None:
    df_dict['ffn_before'] = y_before_ffn
    df_dict['ffn_after'] = y_after_ffn

df = pd.DataFrame(df_dict)
df.to_csv('kan_all_layers_data.csv', index=False)
print("✅ Data saved to kan_all_layers_data.csv")

print("\n✅ Done.")

✅ Model loaded on CPU
Encoder KAN layer: in_features=10, out_features=64
Transformer FFN KAN layer: in_features=64, out_features=16
Pruned 16 basis functions out of 64 (25.00%)
✅ Data saved to kan_all_layers_data.csv

✅ Done.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Загружаем данные
df = pd.read_csv('kan_encoder_data.csv')
x_vals = df['x'].values
y_before = df['before_pruning'].values
y_after = df['after_pruning'].values

# Папка для сохранения
plots_dir = Path("plots")
plots_dir.mkdir(exist_ok=True)

# График ДО
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_vals, y_before, 'b-', linewidth=2)
ax.set_title('Encoder KAN: neuron 0 (before pruning)')
ax.set_xlabel('Input value')
ax.set_ylabel('Output')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(plots_dir / "encoder_before_pruning.png", dpi=150)
plt.close(fig)
print(f"Saved: {plots_dir / 'encoder_before_pruning.png'}")

# График ПОСЛЕ
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_vals, y_after, 'r-', linewidth=2)
ax.set_title('Encoder KAN: neuron 0 (after pruning)')
ax.set_xlabel('Input value')
ax.set_ylabel('Output')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(plots_dir / "encoder_after_pruning.png", dpi=150)
plt.close(fig)
print(f"Saved: {plots_dir / 'encoder_after_pruning.png'}")

# Совмещённый
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_vals, y_before, 'b-', linewidth=2, label='Before')
ax.plot(x_vals, y_after, 'r-', linewidth=2, label='After')
ax.set_title('Encoder KAN: neuron 0 – before vs after pruning')
ax.set_xlabel('Input value')
ax.set_ylabel('Output')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(plots_dir / "encoder_before_after_comparison.png", dpi=150)
plt.close(fig)
print(f"Saved: {plots_dir / 'encoder_before_after_comparison.png'}")

print(f"\n✅ All plots saved in '{plots_dir}' directory.")